In [16]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A.xlsx")
CACHE = Path("../data/interim/sheets_raw")
key = "ID_UC_G0"

if CACHE.exists() and any(CACHE.glob("*.parquet")):
    sheets = {p.stem: pd.read_parquet(p) for p in CACHE.glob("*.parquet")}
else:
    sheets = pd.read_excel(RAW, sheet_name=None)
    CACHE.mkdir(parents=True, exist_ok=True)
    for name, d in sheets.items():
        d.to_parquet(CACHE / f"{name}.parquet")

print(len(sheets), "листов загружено")

17 листов загружено


In [17]:
import pandas as pd
xlsx = Path("../data/raw/GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A.xlsx")
print(pd.ExcelFile(xlsx).sheet_names)

['Description', 'Index', 'GENERAL_CHARACTERISTICS', 'GHSL', 'CLIMATE', 'EMISSIONS', 'EXPOSURE', 'GEOGRAPHY', 'GREENNESS', 'HAZARD_RISK', 'HEALTH', 'INFRASTRUCTURES', 'LULC', 'NATURAL_SYSTEMS', 'SDG', 'SOCIOECONOMIC', 'WATER']


In [18]:
%%time
sheets = pd.read_excel(xlsx, sheet_name=None)  
for name, d in sheets.items():
    print(f"{name:28s} {d.shape}")

Description                  (31, 3)
Index                        (506, 29)
GENERAL_CHARACTERISTICS      (1089, 13)
GHSL                         (1089, 551)
CLIMATE                      (1089, 109)
EMISSIONS                    (1089, 590)
EXPOSURE                     (1089, 1066)
GEOGRAPHY                    (1089, 37)
GREENNESS                    (1089, 63)
HAZARD_RISK                  (1089, 118)
HEALTH                       (1089, 17)
INFRASTRUCTURES              (1089, 17)
LULC                         (1089, 49)
NATURAL_SYSTEMS              (1091, 23)
SDG                          (1089, 43)
SOCIOECONOMIC                (1089, 183)
WATER                        (1089, 28)
CPU times: total: 15.8 s
Wall time: 15.9 s


In [19]:
display(sheets["Description"].head(30))
display(sheets["GENERAL_CHARACTERISTICS"].iloc[:3, :12])
common = set.intersection(*(set(d.columns) for n, d in sheets.items() if n not in ("Description", "Index")))
print(common)


,Name,GHS-UCDB R2024A - GHS Urban Centre Database 2025,Unnamed: 2
0,Dataset ID,GHS-UCDB R2024A,NaN
1,Version,V1.2,NaN
2,Data Last Update,15/05/2026,NaN
3,Alternative name (with version),GHS Urban Centre Database 2024 - GHS Urban Cen...,NaN
4,Accronym,GHS-UCDB 2025 R2024A V1.2,NaN
5,Authors,"Mari Rivero, Ines; Melchiorri, Michele; Florio...",NaN
6,Document Version,V1.1,NaN
7,Document Last Update,30/07/2025,NaN
8,NaN,NaN,NaN
9,Contact point,JRC-GHSL<at>EC.EUROPA.EU,GHSL project related questions


,ID_UC_G0,GC_UCN_MAI_2025,GC_UCN_LIS_2025,GC_CNT_GAD_2025,GC_CNT_UNN_2025,GC_UCA_KM2_2025,GC_POP_TOT_2025,GC_DEV_WIG_2025,GC_DEV_USR_2025,GC_PLS_SCR_2025,GC_UCB_YOB_2025,GC_UCB_YOD_2025
0,5,Rosarito,Rosarito,MÃ©xico,Mexico,29,76707.05999,Upper Middle,Latin America and the Caribbean,High,2015,2030
1,7,Malacatán,MalacatÃ¡n,Guatemala,Guatemala,9,62529.14530,Upper Middle,Latin America and the Caribbean,High,2015,2030
2,8,Ahuachapan,Ahuachapan,El Salvador,El Salvador,19,83851.12663,Upper Middle,Latin America and the Caribbean,High,1980,2030


{'GC_POP_TOT_2025', 'ID_UC_G0', 'GC_UCN_MAI_2025', 'GC_UCA_KM2_2025', 'GC_DEV_USR_2025', 'GC_CNT_GAD_2025', 'GC_DEV_WIG_2025'}


In [20]:
key = "ID_UC_G0"
for name, d in sheets.items():
    if name in ("Description", "Index"):
        continue
    has = key in d.columns
    print(f"{name:24s} ключ_есть={has}  строк={len(d)}  "
          f"уникальных={d[key].nunique() if has else '-'}  "
          f"пропусков={d[key].isna().sum() if has else '-'}")

GENERAL_CHARACTERISTICS  ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
GHSL                     ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
CLIMATE                  ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
EMISSIONS                ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
EXPOSURE                 ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
GEOGRAPHY                ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
GREENNESS                ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
HAZARD_RISK              ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
HEALTH                   ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
INFRASTRUCTURES          ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
LULC                     ключ_есть=True  строк=1089  уникальных=1089  пропусков=0
NATURAL_SYSTEMS          ключ_есть=True  строк=1091  уникальных=1089  пропусков=0
SDG             

In [21]:
ns = sheets["NATURAL_SYSTEMS"]
base = set(sheets["GENERAL_CHARACTERISTICS"][key])
display(ns[~ns[key].isin(base)].iloc[:, :6])        # строки, которых нет в основном листе
display(ns[ns[key].duplicated(keep=False)].iloc[:, :6])   # дубликаты по ключу


,ID_UC_G0,GC_UCN_MAI_2025,GC_CNT_GAD_2025,GC_UCA_KM2_2025,GC_POP_TOT_2025,GC_DEV_WIG_2025


,ID_UC_G0,GC_UCN_MAI_2025,GC_CNT_GAD_2025,GC_UCA_KM2_2025,GC_POP_TOT_2025,GC_DEV_WIG_2025
686,4943,Cúcuta,Colombia,114,817158.1235,Upper Middle
687,4943,Cúcuta,Colombia,114,817158.1235,Upper Middle
858,6621,Reynosa,MÃ©xico,155,616127.2669,Upper Middle
859,6621,Reynosa,MÃ©xico,155,616127.2669,Upper Middle


In [22]:
g = sheets["GENERAL_CHARACTERISTICS"]
txt = g.select_dtypes("object")
bad = {c: txt[c].astype(str).str.contains("Ã|Â", regex=True).sum() for c in txt}
print({c: n for c, n in bad.items() if n})

{'GC_UCN_MAI_2025': np.int64(1), 'GC_UCN_LIS_2025': np.int64(345), 'GC_CNT_GAD_2025': np.int64(183), 'GC_CNT_UNN_2025': np.int64(1)}


C:\Users\Артём\AppData\Local\Temp\ipykernel_10300\63697357.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  txt = g.select_dtypes("object")


In [23]:
ns = sheets["NATURAL_SYSTEMS"]
print("полных дубликатов строк:", ns.duplicated().sum())


полных дубликатов строк: 0


In [29]:
import src, importlib
print(src.__path__)
print(sorted(p.name for p in Path(src.__path__[0]).iterdir()))
importlib.invalidate_caches()
from src.cleaning import fix_mojibake
print(fix_mojibake("MÃ©xico"))

['C:\\Users\\Артём\\Desktop\\аналитиии\\src']
['__init__.py', '__pycache__', 'cleaning.py', 'inspect_raw.py', 'src']


ImportError: cannot import name 'fix_mojibake' from 'src.cleaning' (C:\Users\Артём\Desktop\аналитиии\src\cleaning.py)

In [30]:
import sys
sys.path.insert(0, str(Path("..").resolve()))   # блокнот лежит в notebooks/, src/ на уровень выше

%load_ext autoreload
%autoreload 2
from src.cleaning import fix_mojibake

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ImportError: cannot import name 'fix_mojibake' from 'src.cleaning' (C:\Users\Артём\Desktop\аналитиии\src\cleaning.py)

In [ ]:
def fix_mojibake(s):
    """Чинит только строки с признаками двойного кодирования UTF-8."""
    if not isinstance(s, str) or not any(m in s for m in ("Ã", "Â")):
        return s
    for enc in ("cp1252", "latin-1"):
        try:
            return s.encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return s  # не удалось, оставляем как есть

g = sheets["GENERAL_CHARACTERISTICS"]
cols = ["GC_UCN_MAI_2025", "GC_UCN_LIS_2025", "GC_CNT_GAD_2025", "GC_CNT_UNN_2025"]
fixed = g[cols].map(fix_mojibake)

for c in cols:
    m = fixed[c].fillna("") != g[c].fillna("")
    print(c, "исправлено:", m.sum())
    display(pd.DataFrame({"было": g.loc[m, c], "стало": fixed.loc[m, c]}).drop_duplicates().head(8))

print(fixed.apply(lambda s: s.astype(str).str.contains("Ã|Â")).sum())  # что осталось битым

GC_UCN_MAI_2025 исправлено: 0


,было,стало


GC_UCN_LIS_2025 исправлено: 345


,было,стало
1,MalacatÃ¡n,Malacatán
10,San AndrÃ©s,San Andrés
15,UreÃ±a,Ureña
16,JÃ©rÃ©mie,Jérémie
29,ConcepciÃ³n,Concepción
35,CofradÃ­a,Cofradía
36,LeÃ³n,León
40,ColÃ³n,Colón


GC_CNT_GAD_2025 исправлено: 183


,было,стало
0,MÃ©xico,México
20,CuraÃ§ao,Curaçao


GC_CNT_UNN_2025 исправлено: 1


,было,стало
20,CuraÃ§ao,Curaçao


GC_UCN_MAI_2025    1
GC_UCN_LIS_2025    1
GC_CNT_GAD_2025    0
GC_CNT_UNN_2025    0
dtype: int64


In [ ]:
for name, d in sheets.items():
    if name in ("Description", "Index"):
        continue
    t = d.select_dtypes(include=["object", "string"])
    bad = {c: int(t[c].astype(str).str.contains("Ã|Â").sum()) for c in t.columns}
    bad = {c: n for c, n in bad.items() if n}
    if bad:
        print(name, bad)

GENERAL_CHARACTERISTICS {'GC_UCN_MAI_2025': 1, 'GC_UCN_LIS_2025': 345, 'GC_CNT_GAD_2025': 183, 'GC_CNT_UNN_2025': 1}
GHSL {'GC_UCN_MAI_2025': 1}
CLIMATE {'GC_UCN_MAI_2025': 1}
EMISSIONS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
EXPOSURE {'GC_UCN_MAI_2025': 1}
GEOGRAPHY {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183, 'GE_ECO_CLA_2025': 164}
GREENNESS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
HAZARD_RISK {'GC_UCN_MAI_2025': 1}
HEALTH {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
INFRASTRUCTURES {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
LULC {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
NATURAL_SYSTEMS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 184, 'NS_MPA_NAM_2025': 31, 'NS_TPA_NAM_2025': 192, 'NS_TPA_DES_202': 1}
SDG {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
SOCIOECONOMIC {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
WATER {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}


GENERAL_CHARACTERISTICS {'GC_UCN_MAI_2025': 1, 'GC_UCN_LIS_2025': 345, 'GC_CNT_GAD_2025': 183, 'GC_CNT_UNN_2025': 1}
GHSL {'GC_UCN_MAI_2025': 1}
CLIMATE {'GC_UCN_MAI_2025': 1}
EMISSIONS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
EXPOSURE {'GC_UCN_MAI_2025': 1}
GEOGRAPHY {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183, 'GE_ECO_CLA_2025': 164}
GREENNESS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
HAZARD_RISK {'GC_UCN_MAI_2025': 1}
HEALTH {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
INFRASTRUCTURES {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
LULC {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
NATURAL_SYSTEMS {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 184, 'NS_MPA_NAM_2025': 31, 'NS_TPA_NAM_2025': 192, 'NS_TPA_DES_202': 1}
SDG {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
SOCIOECONOMIC {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}
WATER {'GC_UCN_MAI_2025': 1, 'GC_CNT_GAD_2025': 183}


In [ ]:
sheets["Index"].astype("string").to_parquet(CACHE / "Index.parquet")
print(len(list(CACHE.glob("*.parquet"))), "файлов в кэше")   # ожидаем 16


17 файлов в кэше


In [ ]:
for name, d in sheets.items():
    d.astype("string").to_parquet(CACHE / f"{name}.parquet") if name in ("Index", "Description") \
        else d.to_parquet(CACHE / f"{name}.parquet")
    

In [ ]:
ns = sheets["NATURAL_SYSTEMS"]
dup = ns[ns[key].duplicated(keep=False)].sort_values(key)
diff_cols = [c for c in dup.columns if dup.groupby(key)[c].nunique(dropna=False).gt(1).any()]
print(len(diff_cols), diff_cols)
display(dup[[key] + diff_cols].T)

1 ['NS_TPA_PER_2025']


,686,687,858,859
ID_UC_G0,4943.000000,4943.000000,6621.000000,6621.000000
NS_TPA_PER_2025,0.041406,0.002576,1.428053,0.104751


In [ ]:
for name, d in sheets.items():
    d.astype("string").to_parquet(CACHE / f"{name}.parquet") if name in ("Index", "Description") \
        else d.to_parquet(CACHE / f"{name}.parquet")
    

In [36]:
g = sheets["GENERAL_CHARACTERISTICS"]
cols = ["GC_UCN_MAI_2025", "GC_UCN_LIS_2025", "GC_CNT_GAD_2025", "GC_CNT_UNN_2025"]
fixed = g[cols].map(fix_mojibake)

for c in ["GC_UCN_MAI_2025", "GC_UCN_LIS_2025"]:
    m = fixed[c].astype(str).str.contains("Ã|Â")
    display(g.loc[m, [key, c]].assign(после=fixed.loc[m, c]))

,ID_UC_G0,GC_UCN_MAI_2025,после
510,3496,Santo Ângelo,Santo Ângelo


,ID_UC_G0,GC_UCN_LIS_2025,после
510,3496,Santo Ãngelo,Santo Ângelo


In [37]:
p = Path(src.__path__[0])
print(sorted(x.name for x in (p / "src").iterdir()))

[]


In [38]:
import shutil
shutil.move(str(p / "src" / "cleaning.py"), str(p / "cleaning.py"))
print(sorted(x.name for x in p.iterdir()))

FileNotFoundError: [WinError 2] Не удается найти указанный файл

In [ ]:
importlib.invalidate_caches()
from src.cleaning import fix_mojibake
print(fix_mojibake("MÃ©xico"))   # México

ImportError: cannot import name 'fix_mojibake' from 'src.cleaning' (C:\Users\Артём\Desktop\аналитиии\src\cleaning.py)

In [ ]:
print(repr((Path("..") / "src" / "cleaning.py").read_text(encoding="utf-8")[:300]))

''


In [ ]:
code = '''"""Функции очистки данных GHS-UCDB."""


def fix_mojibake(s):
    """Чинит только строки с признаками двойного кодирования UTF-8."""
    if not isinstance(s, str) or not any(m in s for m in ("Ã", "Â")):
        return s
    for enc in ("cp1252", "latin-1"):
        try:
            return s.encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return s
'''
(Path("..") / "src" / "cleaning.py").write_text(code, encoding="utf-8")

import importlib, sys
importlib.invalidate_caches()
import src.cleaning
importlib.reload(src.cleaning)
from src.cleaning import fix_mojibake
print(fix_mojibake("MÃ©xico"))   # México

México
